# Experimenting MongoDB connections

## Connection

In [30]:
import os

In [13]:
from pymongo.mongo_client import MongoClient

In [29]:
from dotenv import load_dotenv
load_dotenv()

True

In [32]:
uri = os.getenv("MONGO_URI")

In [33]:
# Create a new client and connect to the server
client = MongoClient(uri)

In [34]:
# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [35]:
# Select (or create) the database named 'test_ecom_db' from the MongoDB client
db = client["test_ecom_db"]

# Select (or create) the collection named 'products' inside the database
collection = db["products"]

# Print a message indicating the code reached this point successfully
print("Connected to MongoDB")

Connected to MongoDB


## CREATE

In [19]:
product = {
    "name": "Men Casual Shirt",
    "price": 999,
    "category": "Men"
}

collection.insert_one(product)

InsertOneResult(ObjectId('69edca92f6ee731f504d1128'), acknowledged=True)

In [20]:
products = [
    {
        "name": "Jeans",
        "price": 1499,
        "category": "Men"
    },
    {
        "name": "Kurti",
        "price": 999,
        "category": "Women"
    },
    {
        "name": "Kids Jacket",
        "price": 1299,
        "category": "Kids"
    },
]

collection.insert_many(products)

InsertManyResult([ObjectId('69edca93f6ee731f504d1129'), ObjectId('69edca93f6ee731f504d112a'), ObjectId('69edca93f6ee731f504d112b')], acknowledged=True)

## READ

In [21]:
find_product = collection.find_one({"name": "Jeans"})

print(find_product)

{'_id': ObjectId('69edc8a3f6ee731f504d1124'), 'name': 'Jeans', 'price': 1499, 'category': 'Men'}


In [27]:
men_products = collection.find({
    "category": "Kids"
})

for p in men_products:
    print(p)

{'_id': ObjectId('69edc8a3f6ee731f504d1126'), 'name': 'Kids Jacket', 'price': 1599, 'category': 'Kids'}


## UPDATE

In [24]:
result = collection.update_one(
    # FILTER: Find the document where the 'name' field is "Jeans"
    {"name": "Jeans"},
    # UPDATE: Set (update) the 'price' field to 1399
    {"$set": {"price": 1399}}
)

print(result.matched_count)    # Number of documents matched`

print(result.modified_count)   # Number of documents updated`

1
1


In [26]:
[p for p in collection.find({"name": "Jeans"})]

[{'_id': ObjectId('69edc8a3f6ee731f504d1124'),
  'name': 'Jeans',
  'price': 1399,
  'category': 'Men'}]

`{"$set": {"price": 1399}}`
* `$set` updates only the specified field.
* If `price` already exists → its value is replaced with 1399.
* If `price` does NOT exist → MongoDB creates it.

✅ Only the `price` field is changed

❌ Other fields (`name`, `category`, etc.) remain unchanged

❌ Without `$set`, MongoDB would replace the entire document

### **❌ Wrong (dangerous) example:**

`collection.update_one({"name": "Jeans"},{"price": 1399}   # Replaces whole document!)`

**Other operators are:**
1) `$inc` — Increment a numeric value
* What it does
    * Increases or decreases a number field
    * Very useful for prices, quantity, counters, stock, likes, etc.

2) `$unset` — Remove a field completely
* What it does
    * Deletes a field from the document
    * Does NOT delete the entire document

3) `$push` — Add an item to an array
* What it does
    * Adds one value to an array field
    * Creates the array if it does not exist

& more

🔁 $push vs $addToSet (important difference)

`{"$push": {"sizes": "M"}}        # Allows duplicates`

`{"$addToSet": {"sizes": "M"}}    # Prevents duplicates`

## DELETE

In [28]:
collection.delete_one({
    "name": "Kids Jacket"
})

DeleteResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000112'), 'opTime': {'ts': Timestamp(1777198006, 2), 't': 274}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1777198006, 2), 'signature': {'hash': b'\x96\xa6b\xf6\x9d\xeb+\x8c!\xa4\xc8r\xab\x8b5\xc8\xd5\xe0\xa4\xed', 'keyId': 7594861961678946318}}, 'operationTime': Timestamp(1777198006, 2)}, acknowledged=True)